In [ ]:
# Loading data
import pandas as pd

df = pd.read_csv("Wacom_Sales_5_Years.csv")

df.shape

# Data type conversion
df["Date"] = pd.to_datetime(df["Date"])
df["Region"] = df["Region"].str.strip()
df["Product_Category"] = df["Product_Category"].str.strip()
df["Product_Name"] = df["Product_Name"].str.strip()
df["Service_Type"] = df["Service_Type"].str.strip()

print(df.dtypes)

# Missing values
print(df.isnull().sum())
print(df.isnull().sum().sum())

# Duplicates
print(df.duplicated().sum())
print(df.duplicated(subset=["Date", "Region", "Product_Name"]).sum())

# Outlier detection
Q1 = df["Total_Revenue_USD"].quantile(0.25)
Q3 = df["Total_Revenue_USD"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(Q1)
print(Q3)
print(IQR)

outliers = df[(df["Total_Revenue_USD"] < lower_bound) |
              (df["Total_Revenue_USD"] > upper_bound)]

print(len(outliers))

df["Revenue_Outlier_Flag"] = (
    (df["Total_Revenue_USD"] < lower_bound) | (df["Total_Revenue_USD"] > upper_bound)
).astype(int)

print(df["Revenue_Outlier_Flag"].value_counts())

# Feature engineering
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day_Of_Week"] = df["Date"].dt.dayofweek

print(df.head())

df["Revenue_per_Unit"] = (df["Total_Revenue_USD"] / df["Units_Sold"]).round(2)

monthly = df.groupby(["Year", "Month"])["Total_Revenue_USD"].sum().reset_index()
monthly.columns = ["Year", "Month", "Monthly_Revenue"]

print(monthly.head())

monthly["Next_Month_Revenue"] = monthly["Monthly_Revenue"].shift(-1)

df = df.merge(monthly[["Year", "Month", "Next_Month_Revenue"]], on=["Year", "Month"], how="left")

print(df.head())

# Final validation
print(df.shape)
print(df.isnull().sum().sum())
df.to_csv("cleaned_wacom.csv", index=False)

Date                   datetime64[ns]
Region                         object
Product_Category               object
Product_Name                   object
Service_Type                   object
Units_Sold                      int64
Product_Revenue_USD           float64
Service_Revenue_USD           float64
Total_Revenue_USD             float64
dtype: object
Date                   0
Region                 0
Product_Category       0
Product_Name           0
Service_Type           0
Units_Sold             0
Product_Revenue_USD    0
Service_Revenue_USD    0
Total_Revenue_USD      0
dtype: int64
0
0
0
93379.2875
265406.055
172026.7675
2
Revenue_Outlier_Flag
0    1824
1       2
Name: count, dtype: int64
        Date         Region Product_Category         Product_Name  \
0 2021-01-01           APAC       Pen Tablet       Wacom Intuos M   
1 2021-01-02           APAC    Mobile Studio  MobileStudio Pro 16   
2 2021-01-03  North America    Mobile Studio  MobileStudio Pro 13   
3 2021-01-04  North A